In [1]:
import pandas as pd

In [2]:
ml_data = pd.read_csv("../../data_processed/combined_ml_data.csv")
ml_data.head()

,variety_id,variety_name,crop_name,trait_name,trait_type,location_id,town,lat,lon,country,...,humidity,rainfall,condition,month,year,tpe_name,tpe_description,stressors,yield,adoption_rate
0,1,Mucuvea(2015),sorghum,APHIDS_RESISTANCE,Desired,11,Maputo,-25.9686,32.5804,Mozambique,...,57.0,-0.1,broken clouds,December,2025,ARID,"Extremely dry areas with minimal rainfall, oft...","['DROUGHT', 'HEAT', 'STRIGA', 'SALINITY']",NaN,NaN
1,1,Mucuvea(2015),sorghum,APHIDS_RESISTANCE,Desired,11,Maputo,-25.9686,32.5804,Mozambique,...,59.0,0.1,broken clouds,November,2025,ARID,"Extremely dry areas with minimal rainfall, oft...","['DROUGHT', 'HEAT', 'STRIGA', 'SALINITY']",NaN,NaN
2,1,Mucuvea(2015),sorghum,APHIDS_RESISTANCE,Desired,11,Maputo,-25.9686,32.5804,Mozambique,...,57.0,-0.1,broken clouds,October,2025,ARID,"Extremely dry areas with minimal rainfall, oft...","['DROUGHT', 'HEAT', 'STRIGA', 'SALINITY']",NaN,NaN
3,1,Mucuvea(2015),sorghum,APHIDS_RESISTANCE,Desired,11,Maputo,-25.9686,32.5804,Mozambique,...,59.0,0.1,broken clouds,September,2025,ARID,"Extremely dry areas with minimal rainfall, oft...","['DROUGHT', 'HEAT', 'STRIGA', 'SALINITY']",NaN,NaN
4,1,Mucuvea(2015),sorghum,APHIDS_RESISTANCE,Desired,11,Maputo,-25.9686,32.5804,Mozambique,...,57.0,-0.1,broken clouds,August,2025,ARID,"Extremely dry areas with minimal rainfall, oft...","['DROUGHT', 'HEAT', 'STRIGA', 'SALINITY']",NaN,NaN


## Data review for cleaning

In [3]:
print(ml_data.isnull().sum())

variety_id              0
variety_name            0
crop_name               0
trait_name              0
trait_type              0
location_id             0
town                    0
lat                     0
lon                     0
country                 0
temperature             0
humidity                0
rainfall                0
condition               0
month                   0
year                    0
tpe_name                0
tpe_description         0
stressors               0
yield              144552
adoption_rate      144552
dtype: int64


In [4]:
ml_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144552 entries, 0 to 144551
Data columns (total 21 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   variety_id       144552 non-null  int64  
 1   variety_name     144552 non-null  object 
 2   crop_name        144552 non-null  object 
 3   trait_name       144552 non-null  object 
 4   trait_type       144552 non-null  object 
 5   location_id      144552 non-null  int64  
 6   town             144552 non-null  object 
 7   lat              144552 non-null  float64
 8   lon              144552 non-null  float64
 9   country          144552 non-null  object 
 10  temperature      144552 non-null  float64
 11  humidity         144552 non-null  float64
 12  rainfall         144552 non-null  float64
 13  condition        144552 non-null  object 
 14  month            144552 non-null  object 
 15  year             144552 non-null  int64  
 16  tpe_name         144552 non-null  obje

#### Drop null columns

In [5]:
# Drop the 'yield' and 'adoption_rate' columns
ml_data = ml_data.drop(columns=['yield', 'adoption_rate'])
ml_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144552 entries, 0 to 144551
Data columns (total 19 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   variety_id       144552 non-null  int64  
 1   variety_name     144552 non-null  object 
 2   crop_name        144552 non-null  object 
 3   trait_name       144552 non-null  object 
 4   trait_type       144552 non-null  object 
 5   location_id      144552 non-null  int64  
 6   town             144552 non-null  object 
 7   lat              144552 non-null  float64
 8   lon              144552 non-null  float64
 9   country          144552 non-null  object 
 10  temperature      144552 non-null  float64
 11  humidity         144552 non-null  float64
 12  rainfall         144552 non-null  float64
 13  condition        144552 non-null  object 
 14  month            144552 non-null  object 
 15  year             144552 non-null  int64  
 16  tpe_name         144552 non-null  obje

### ML Data preprocessing
* At this stage, we are preparing the data for machine learning by cleaning and transforming it into a format that algorithms can understand. This involves removing unnecessary columns, converting categorical data into numerical format using one-hot encoding, and scaling numerical features (like temperature and rainfall) to ensure they are on a similar scale. This preparation is crucial for accurate and effective machine learning analysis, which will help us recommend the best crop varieties for specific environments.

### Encoding categorical variables

* One-hot encoding is a technique used to convert categorical data (like names, labels, or categories) into a numerical format that machine learning algorithms can process. It creates new binary columns for each category in a variable, where a value of 1 indicates the presence of that category, and 0 indicates its absence. This prevents the algorithm from misinterpreting categorical data as having an ordinal relationship (e.g., assuming "2" is greater than "1").

In [6]:
ml_data_encoded = pd.get_dummies(ml_data, columns=['trait_name', 'tpe_name', 'condition', 'month'])
ml_data_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144552 entries, 0 to 144551
Columns: 140 entries, variety_id to month_September
dtypes: bool(125), float64(5), int64(3), object(7)
memory usage: 33.8+ MB


In [8]:
# Step 3: Check for missing values and drop them if any
ml_data_encoded = ml_data_encoded.dropna()

### Scaling numerical features
* Scaling numerical features (like temperature and rainfall) to ensure they are on a similar scale.
* StandardScaler is used to standardize numerical features by removing the mean and scaling to unit variance.
* fit_transform: This method computes the mean and standard deviation of each feature (fit) and then scales the data so that each feature has a mean of 0 and a standard deviation of 1 (transform).
* The scaled values replace the original values in the ml_data_encoded DataFrame.


In [10]:
from sklearn.preprocessing import StandardScaler

numerical_features = ['temperature', 'humidity', 'rainfall', 'lat', 'lon']
scaler = StandardScaler()
ml_data_encoded[numerical_features] = scaler.fit_transform(ml_data_encoded[numerical_features])
ml_data_encoded.head()

,variety_id,variety_name,crop_name,trait_type,location_id,town,lat,lon,country,temperature,...,month_December,month_February,month_January,month_July,month_June,month_March,month_May,month_November,month_October,month_September
0,1,Mucuvea(2015),sorghum,Desired,11,Maputo,-2.727707,0.860637,Mozambique,-0.089199,...,True,False,False,False,False,False,False,False,False,False
1,1,Mucuvea(2015),sorghum,Desired,11,Maputo,-2.727707,0.860637,Mozambique,0.333439,...,False,False,False,False,False,False,False,True,False,False
2,1,Mucuvea(2015),sorghum,Desired,11,Maputo,-2.727707,0.860637,Mozambique,0.122120,...,False,False,False,False,False,False,False,False,True,False
3,1,Mucuvea(2015),sorghum,Desired,11,Maputo,-2.727707,0.860637,Mozambique,-0.089199,...,False,False,False,False,False,False,False,False,False,True
4,1,Mucuvea(2015),sorghum,Desired,11,Maputo,-2.727707,0.860637,Mozambique,0.333439,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# Prepare features for clustering
X = ml_data_encoded.drop(columns=['variety_id', 'variety_name', 'crop_name', 'location_id', 'town', 'country', 'year'])

# Display the prepared data
print(X.head())